# Stage 1 · Fringe pattern tests
**SOLAR Lab · Deflectometry backbone** — Diego Bethancourth · Advisor: Dr. Xiangyu Guo

This notebook tests the first block shared by every deflectometry pipeline in the lab: the sinusoidal fringe pattern shown on the screen.

| Pipeline | What it displays | Where the fringe enters |
|---|---|---|
| Phase-shifting deflectometry (PMD, Spring 2026 WK1) | N patterns per axis, each shifted by 2π/N | Pixel-by-pixel phase map → slope → surface |
| SMOTS (Choi et al. 2017) | One static pattern | Fourier ratio before/after a tilt → tilt angle |

The SMOTS frame is step k = 0 of the PMD sequence, so one generator serves both.

**Lab convention (fixed):**

$$I_k(x) = A + B\cos\!\left(\frac{2\pi x}{P} + \delta_k\right),\qquad \delta_k = \frac{2\pi k}{N},\; k = 0,\dots,N-1$$

$$\varphi = \operatorname{atan2}\!\left(-\sum_k I_k\sin\delta_k,\; \sum_k I_k\cos\delta_k\right)$$

For N = 4 this is the WK1 MATLAB line `atan2(I4 - I2, I1 - I3)`.

| Symbol | Meaning | Unit |
|---|---|---|
| I_k | grey level of frame k | 0–255 |
| A | bias (mean level) | grey levels |
| B | amplitude (modulation) | grey levels |
| P | fringe period | screen px |
| N | number of phase steps | — |
| δ_k | phase step of frame k | rad |
| φ | wrapped phase | rad, (−π, π] |

Each test below has a known answer and needs no hardware. Sections 3–11 run one test group each; section 12 exports the frames for the bench; section 13 runs the full automated suite.

## 1 · Libraries
- **numpy** — all computation (the only dependency of the method, as in the `smots` package).
- **matplotlib** — plots only.
- **Pillow (PIL)** — writes the PNG frames that go on the screen.
- **pytest** — runs the automated test suite in section 13.

In [ ]:
!pip -q install pytest pillow
import numpy as np
import matplotlib.pyplot as plt

## 2 · Get the code
The module `deflecto/fringe.py` and the tests `tests/test_fringe.py` live in the lab repository next to `smots/`.
The cell clones the repository when running in Colab. If the folder is not yet on GitHub, upload `fringe.py` and `test_fringe.py` with the file panel on the left instead; the cell arranges them into the expected folders.

In [ ]:
import os, sys, shutil, subprocess

REPO = "https://github.com/diegobethancourth/deflectometry-mirror-array"
if not os.path.exists("deflecto/fringe.py"):
    if not os.path.exists("deflectometry-mirror-array"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO])
    if os.path.exists("deflectometry-mirror-array/deflecto/fringe.py"):
        os.chdir("deflectometry-mirror-array")
    elif os.path.exists("fringe.py"):                     # uploaded by hand
        os.makedirs("deflecto", exist_ok=True); os.makedirs("tests", exist_ok=True)
        shutil.copy("fringe.py", "deflecto/fringe.py")
        open("deflecto/__init__.py", "a").close()
        if os.path.exists("test_fringe.py"):
            shutil.copy("test_fringe.py", "tests/test_fringe.py")

sys.path.insert(0, os.getcwd())
from deflecto.fringe import (FringeSpec, generate, retrieve_phase, retrieve_phase_wk1,
                             display, wrap, expected_phase_noise, fit_profile, TWO_PI)
print("fringe module loaded from", os.path.abspath("deflecto/fringe.py"))

## 3 · Bench parameters
Screen: ViewSonic VX1655-4K, W = 344 mm, N_x = 3840 px → pixel pitch p = W / N_x.
Fringe period P = 30 px (the value used in Choi et al. Sec. 3.1 and in `01_shear_theorem.py`); physical period P_f = P · p.
The test line has N_s = 510 samples = 17 whole periods, so the FFT sees no partial period at the window edge.

> The pixel pitch is a datasheet value. It must be measured under a microscope before any number from the bench is reported (smots README, "Before this touches hardware").

In [ ]:
SCREEN_W_MM, SCREEN_NX, SCREEN_NY = 344.0, 3840, 2160
p_mm = SCREEN_W_MM / SCREEN_NX              # p = W / Nx
P_px = 30.0                                 # P, screen px per fringe
W, H = 510, 8                               # test image: 17 periods x 8 rows

spec4 = FringeSpec(period_px=P_px, n_steps=4)   # lab default: A = 128, B = 100
print(f"pixel pitch p   = {p_mm*1e3:.1f} um")
print(f"fringe period   = {P_px:.0f} px = {spec4.period_mm(p_mm):.3f} mm")
print(f"levels          = {spec4.bias - spec4.amplitude:.0f} .. {spec4.bias + spec4.amplitude:.0f} (margin kept from 0 and 255)")
print(f"phase steps     = {np.degrees(spec4.deltas)} deg")

## 4 · Test A1 — the four frames
`generate(spec, width, height, axis)` returns an array (N, height, width) of uint8 — exactly what is written to PNG.
Expected: four cosines of identical shape, each moved a quarter period (7.5 px) relative to the previous one.

In [ ]:
frames = generate(spec4, W, H, axis="x")
x = np.arange(W)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
for k in range(4):
    ax[0].plot(x[:90], frames[k, 0, :90], label=f"k={k}, δ={k*90}°")
ax[0].set(xlabel="screen pixel x", ylabel="grey level", title="One row of each frame (first 3 periods)")
ax[0].legend(fontsize=8)
ax[1].imshow(frames[0], cmap="gray", aspect="auto", vmin=0, vmax=255)
ax[1].set(title="Frame k = 0 (vertical stripes → x-slope)")
plt.tight_layout(); plt.show()
print("dtype", frames.dtype, "| min", frames.min(), "| max", frames.max())

## 5 · Test A2 — the period is exact
FFT of one row: the peak index must equal N_s / P = 510 / 30 = 17.

In [ ]:
row = frames[0, 0].astype(float)
spectrum = np.abs(np.fft.rfft(row - row.mean()))
k_peak = int(np.argmax(spectrum))
print(f"peak bin = {k_peak}   expected = {W / P_px:.0f}   ->", "PASS" if k_peak == round(W / P_px) else "FAIL")
plt.figure(figsize=(7, 2.5)); plt.stem(spectrum[:40]); plt.xlabel("FFT bin"); plt.title("Spectrum of one row"); plt.show()

## 6 · Test B1 — phase round trip
Retrieve φ from the four uint8 frames and compare with the true phase 2πx/P.
Without rounding the error is ~10⁻¹⁵ rad. With 8-bit rounding a few milliradians remain — the floor set by the display itself.

In [ ]:
truth = wrap(TWO_PI * x / P_px)
exact = retrieve_phase(generate(spec4, W, H, quantise=False))
quant = retrieve_phase(frames)
err_exact = wrap(exact.phase[0] - truth)
err_quant = wrap(quant.phase[0] - truth)
print(f"max error, no rounding : {np.abs(err_exact).max():.1e} rad")
print(f"max error, 8-bit frames: {np.abs(err_quant).max():.4f} rad  "
      f"= {np.abs(err_quant).max()/TWO_PI*P_px*p_mm*1e3:.2f} um on the screen")
print(f"recovered B = {quant.modulation.mean():.2f} (set 100), A = {quant.bias.mean():.2f} (set 128)")

fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(x, quant.phase[0]); ax[0].set(title="Wrapped phase φ(x)", xlabel="x (px)", ylabel="rad")
ax[1].plot(x, err_quant * 1e3); ax[1].set(title="Error from 8-bit rounding", xlabel="x (px)", ylabel="mrad")
plt.tight_layout(); plt.show()

## 7 · Test B2 — the WK1 MATLAB formula is the N = 4 case
Spring 2026 WK1 used `phi = atan2(I4 - I2, I1 - I3)` with δ = 0, π/2, π, 3π/2.
Expansion: I₁ − I₃ = 2B cos φ and I₄ − I₂ = 2B sin φ, so the two formulas must agree to machine precision.

**Sign trap:** generating with −δ_k (as written in the repository README, `cos(2πx/n − 2πk/N)`) and decoding with +δ_k negates every phase and therefore every slope. The second check shows it.

In [ ]:
f = frames.astype(float)
wk1 = retrieve_phase_wk1(f[0], f[1], f[2], f[3])
diff = np.abs(wrap(wk1 - quant.phase)).max()
print(f"WK1 formula vs general N-step: max difference {diff:.1e} rad ->", "PASS" if diff < 1e-12 else "FAIL")

flipped = retrieve_phase(f[[0, 3, 2, 1]])          # frames played with -delta
print(f"reversed step direction: φ_flipped + φ = {np.abs(wrap(flipped.phase + quant.phase)).max():.1e}  (the sign flips)")

## 8 · Test B3 — camera noise against the error budget
Expected phase noise for white camera noise σ_I:  σ_φ = √(2/N) · σ_I / B  (error-budget relation, repository README).
Converted to the screen: δx_s = σ_φ · P_f / 2π.

In [ ]:
rng = np.random.default_rng(1)
print(" N  sigma_I   predicted     measured    screen error")
for n in (3, 4, 8):
    s = FringeSpec(period_px=P_px, n_steps=n)
    for sig in (1.0, 2.0, 4.0):
        noisy = generate(s, W, 200, quantise=False) + rng.normal(0, sig, (n, 200, W))
        e = wrap(retrieve_phase(noisy).phase - truth[None, :])
        pred = expected_phase_noise(s, sig)
        print(f"{n:2d}   {sig:4.1f}   {pred*1e3:7.2f} mrad  {e.std()*1e3:7.2f} mrad   {e.std()/TWO_PI*s.period_mm(p_mm)*1e3:6.2f} um")

## 9 · Test C1 — display gamma
A screen does not emit light proportional to the grey level sent; it emits ≈ 255·(v/255)^γ with γ ≈ 2.2. That distorts the cosine into harmonics.
An N-step algorithm rejects harmonics up to order N − 2: **3-step is hit by the 2nd harmonic, 4-step only by the 3rd**.
Pre-compensation sends 255·(I/255)^(1/γ), so the light emitted is the intended cosine.

Expected: 3-step without correction fails badly; with correction both return to the 8-bit floor.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
for n, a in zip((3, 4), ax):
    raw  = retrieve_phase(display(generate(FringeSpec(n_steps=n), W, H), 2.2))
    comp = retrieve_phase(display(generate(FringeSpec(n_steps=n, gamma=2.2), W, H), 2.2))
    e_raw, e_comp = wrap(raw.phase[0] - truth), wrap(comp.phase[0] - truth)
    a.plot(x, e_raw * 1e3, label=f"no correction  (max {np.abs(e_raw).max()*1e3:.1f} mrad)")
    a.plot(x, e_comp * 1e3, label=f"pre-compensated (max {np.abs(e_comp).max()*1e3:.1f} mrad)")
    a.set(title=f"{n}-step on a γ = 2.2 display", xlabel="x (px)", ylabel="phase error (mrad)"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10 · Test C2 — headroom (WK1 pattern vs lab standard)
WK1 generated `0.5 + 0.5·cos(...)`, which in 8 bits spans exactly 0–255. Any extra exposure or gain then flattens the peaks — the clipping recorded in the WK1 ISO study.
The lab standard (A = 128, B = 100 → 28–228) keeps a margin on both sides. The cell over-exposes both patterns by 10 % (camera gain or exposure slightly high).

Note also that WK1 set `L = max(screenW, screenH)`, i.e. **one period across the whole screen**. That is suitable for a first demonstration but gives far fewer than the ≥ 60 fringes across the array required by the WK3 sizing study; the lab standard specifies P in screen pixels.

In [ ]:
wk1_spec = FringeSpec(bias=127.5, amplitude=127.5, n_steps=4)
for name, s in (("WK1 0.5+0.5cos", wk1_spec), ("lab standard  ", spec4)):
    captured = np.clip(1.1 * generate(s, W, 1)[0, 0].astype(float), 0, 255)
    fit = fit_profile(captured)
    print(f"{name}: clipped {100*fit.clipped_fraction:5.1f} %   R² {fit.r_squared:.4f}")

## 11 · Test D — quality check of a captured row
`fit_profile(row)` fits A + B cos(2πx/P_cam + φ) to one row of a camera image. It finds the apparent period in **camera pixels** from the data, so the magnification is not needed.
It replaces the WK1 Pearson score, which compared against a wave of assumed phase and therefore penalised a correct capture that happened to be shifted.

**Acceptance criteria proposed for the bench** (to be confirmed on the first captures):

| Quantity | Target |
|---|---|
| R² | ≥ 0.98 |
| Clipped samples | 0 % |
| Visibility B/A | ≥ 0.3 |
| Camera px per fringe | ≥ 10 (camera report, Step 2) |

Use on a real capture: `row = np.asarray(PIL.Image.open("capture.png").convert("L"), float)[row_index]`.

In [ ]:
rng = np.random.default_rng(0)
xc = np.arange(800.0)
P_cam = 12.2                                   # camera px per fringe at z = 432 mm (camera report)
cases = {
    "good capture"        : 120 + 60 * np.cos(TWO_PI * xc / P_cam + 1.0) + rng.normal(0, 1.5, xc.size),
    "over-exposed"        : np.clip(200 + 90 * np.cos(TWO_PI * xc / P_cam) + rng.normal(0, 1.5, xc.size), 0, 255),
    "low contrast / noisy": 120 + 8 * np.cos(TWO_PI * xc / P_cam) + rng.normal(0, 6, xc.size),
}
fig, ax = plt.subplots(1, 3, figsize=(14, 2.8))
for a, (name, r) in zip(ax, cases.items()):
    fit = fit_profile(r)
    ok = fit.r_squared >= 0.98 and fit.clipped_fraction == 0 and fit.contrast >= 0.3
    print(f"{name:22s} {fit.report()}  -> {'ACCEPT' if ok else 'REJECT'}")
    a.plot(xc[:80], r[:80]); a.set(title=name, ylim=(0, 260))
plt.tight_layout(); plt.show()

## 12 · Export the frames for the screen
Full-resolution PNGs for the 3840 × 2160 monitor, 4 steps × 2 axes, plus the SMOTS static frame.

Rules for displaying them on the bench:
1. Show at **100 % scale, full screen**, with OS display scaling at 100 % — any resampling changes the period.
2. Use an image viewer with no colour management or sharpening.
3. Keep `gamma=None` until the display gamma has been measured; then regenerate with `FringeSpec(gamma=<measured>)`.

In [ ]:
from PIL import Image
OUT = "fringes_3840x2160"
os.makedirs(OUT, exist_ok=True)
for axis in ("x", "y"):
    seq = generate(spec4, SCREEN_NX, SCREEN_NY, axis=axis)
    for k, frame in enumerate(seq):
        Image.fromarray(frame).save(f"{OUT}/P{int(P_px)}_N4_{axis}_k{k}.png")
Image.fromarray(generate(FringeSpec(period_px=P_px, n_steps=1), SCREEN_NX, SCREEN_NY)[0]).save(f"{OUT}/P{int(P_px)}_SMOTS_x.png")

# lossless check: what was written is what was generated
back = np.asarray(Image.open(f"{OUT}/P{int(P_px)}_N4_x_k0.png"))
print(sorted(os.listdir(OUT)))
print("PNG round trip identical:", np.array_equal(back, generate(spec4, SCREEN_NX, SCREEN_NY, axis="x")[0]))
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files; files.download(OUT + ".zip")
except ImportError:
    print("zip written to", os.path.abspath(OUT + ".zip"))

## 13 · Full automated suite
Runs every test in `tests/test_fringe.py` (33 checks). All must pass before the fringe stage is accepted as the lab backbone. The SMOTS-compatibility test is skipped when `smots/` is not next to `deflecto/`.

In [ ]:
!python -m pytest -q tests/test_fringe.py